# Weekly premium regression (synthetic DGP)

Target is weekly premium INR clipped to `PREMIUM.BASE`–`PREMIUM.MAX`. Features are stylized (month, historical events, forecast/aqi/social risk, deliveries). This does **not** replace [lib/ml/premium-calc.ts](../../lib/ml/premium-calc.ts); it is a sandbox to later swap in real weekly cohort features.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(ROOT / "scripts"))

from synthetic_data import generate_premium_factor_rows
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score

df = generate_premium_factor_rows(8000)
df.describe()

In [ ]:
features = ["month", "hist_events_4w", "forecast_risk", "aqi_risk", "social_risk", "avg_daily_deliveries"]
X, y = df[features], df["weekly_premium_inr"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("reg", GradientBoostingRegressor(random_state=42, max_depth=4, n_estimators=150, learning_rate=0.06)),
])
pipe.fit(X_train, y_train)
pred = pipe.predict(X_test)
print("holdout R2", r2_score(y_test, pred))
print("holdout MAE INR", mean_absolute_error(y_test, pred))

In [ ]:
import joblib
ROOT.joinpath("artifacts").mkdir(parents=True, exist_ok=True)
ROOT.joinpath("data/synthetic").mkdir(parents=True, exist_ok=True)
joblib.dump(pipe, ROOT / "artifacts" / "premium_weekly.joblib")
df.to_csv(ROOT / "data" / "synthetic" / "premium_weekly.csv", index=False)